In [55]:
import os

DATA_PATH = "data/ml-latest-small"

print(os.listdir(DATA_PATH))

['.ipynb_checkpoints', 'links.csv', 'movies.csv', 'ratings.csv', 'README.txt', 'tags.csv']


In [56]:
import pandas as pd

movies = pd.read_csv("data/ml-latest-small/movies.csv")
ratings = pd.read_csv("data/ml-latest-small/ratings.csv")

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [57]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [58]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

1) Collaborative Filtering

In [59]:
def recommend_movies(user_id, top_n=5):
    user_index = user_id - 1

    sim_scores = user_similarity[user_index]

    similar_users = sim_scores.argsort()[::-1][1:6]

    user_ratings = user_movie_matrix.iloc[user_index]

    scores = {}

    for u in similar_users:
        for movie_id, rating in user_movie_matrix.iloc[u].items():
            if user_ratings[movie_id] == 0:
                scores[movie_id] = scores.get(movie_id, 0) + rating

    top_movies = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    result = movies[movies["movieId"].isin([m[0] for m in top_movies])]

    return result

In [60]:
recommend_movies(user_id=1, top_n=5)

,movieId,title,genres
474,541,Blade Runner (1982),Action|Sci-Fi|Thriller
507,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
659,858,"Godfather, The (1972)",Crime|Drama
902,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
1211,1610,"Hunt for Red October, The (1990)",Action|Adventure|Thriller


2) Content-Based Filtering

In [61]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("data/ml-latest-small/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [62]:
movies["genres"] = movies["genres"].str.replace("|", " ")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [63]:
vectorizer = CountVectorizer()

genre_matrix = vectorizer.fit_transform(movies["genres"])

In [64]:
cosine_sim = cosine_similarity(genre_matrix)

In [65]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

In [66]:
def recommend_movie_content(title, top_n=5):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]

    return movies.iloc[movie_indices][["title", "genres"]]

In [67]:
recommend_movie_content("Toy Story (1995)", 5)

,title,genres
1706,Antz (1998),Adventure Animation Children Comedy Fantasy
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy
3568,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy


3) Hybrid Recommender

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("data/ml-latest-small/movies.csv")
ratings = pd.read_csv("data/ml-latest-small/ratings.csv")

In [71]:
movies["genres"] = movies["genres"].str.replace("|", " ")

vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies["genres"])

content_sim = cosine_similarity(genre_matrix)

In [72]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

user_similarity = cosine_similarity(user_movie_matrix)

In [73]:
def hybrid_recommend(user_id, movie_title, top_n=5):

    # ---------- Content-Based ----------
    movie_idx = movies[movies["title"] == movie_title].index[0]
    content_scores = list(enumerate(content_sim[movie_idx]))

    # ---------- Collaborative ----------
    user_idx = user_id - 1
    sim_users = user_similarity[user_idx].argsort()[::-1][1:6]

    user_ratings = user_movie_matrix.iloc[user_idx]

    collab_scores = {}

    for u in sim_users:
        for movie_id, rating in user_movie_matrix.iloc[u].items():
            if user_ratings[movie_id] == 0:
                collab_scores[movie_id] = collab_scores.get(movie_id, 0) + rating

    # ---------- ترکیب ----------
    hybrid_scores = {}

    for i, score in content_scores:
        movie_id = movies.iloc[i]["movieId"]

        content_score = score
        collab_score = collab_scores.get(movie_id, 0)

        hybrid_scores[movie_id] = content_score + collab_score

    # ---------- خروجی ----------
    top_movies = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    result = movies[movies["movieId"].isin([m[0] for m in top_movies])]

    return result[["title", "genres"]]

In [74]:
hybrid_recommend(
    user_id=1,
    movie_title="Toy Story (1995)",
    top_n=5
)

,title,genres
474,Blade Runner (1982),Action Sci-Fi Thriller
507,Terminator 2: Judgment Day (1991),Action Sci-Fi
659,"Godfather, The (1972)",Crime Drama
902,Aliens (1986),Action Adventure Horror Sci-Fi
1211,"Hunt for Red October, The (1990)",Action Adventure Thriller
